In [ ]:
import pandas as pd
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences

# Load AG News Dataset
# Assume you have files '../data/ag_news_csv/train.csv'  '../data/ag_news_csv/test.csv'
train_df = pd.read_csv('../data/ag_news_csv/train.csv')
test_df = pd.read_csv('../data/ag_news_csv/test.csv')

# Inspect data
print(train_df.head())

# Split data into text and labels
x_train = train_df['Description'].values
y_train = train_df['Class Index'].values
x_test = test_df['Description'].values
y_test = test_df['Class Index'].values

# AG News labels are 1-4; shift to 0-3 for sparse_categorical_crossentropy
y_train = y_train - 1
y_test  = y_test  - 1

# Define maximum words and sequence length
max_words = 10000
max_len = 100  # AG News has shorter text than IMDB

# Tokenize text
tokenizer = Tokenizer(num_words=max_words)
tokenizer.fit_on_texts(x_train)

# Convert text to sequences
x_train = tokenizer.texts_to_sequences(x_train)
x_test = tokenizer.texts_to_sequences(x_test)

# Pad sequences to uniform length
x_train = pad_sequences(x_train, maxlen=max_len)
x_test = pad_sequences(x_test, maxlen=max_len)

print(f'Text example (tokenized): {x_train[0]}')
print(f'Label: {y_train[0]}')

In [ ]:
import pandas as pd
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences

# Load AG News Dataset
train_df = pd.read_csv('../data/ag_news_csv/train.csv')
test_df = pd.read_csv('../data/ag_news_csv/test.csv')

# Combine data from columns "Title"  "Description"
train_df['combined_text'] = train_df['Title'] + ' ' + train_df['Description']
test_df['combined_text'] = test_df['Title'] + ' ' + test_df['Description']

# Inspect data
print(train_df[['Title', 'Description', 'combined_text']].head())

# Split data into text and labels
x_train = train_df['combined_text'].values
y_train = train_df['Class Index'].values
x_test = test_df['combined_text'].values
y_test = test_df['Class Index'].values

# AG News labels are 1-4; shift to 0-3 for sparse_categorical_crossentropy
y_train = y_train - 1
y_test  = y_test  - 1

# Define maximum words and sequence length
max_words = 10000
max_len = 100  # AG News has shorter text than IMDB

# Tokenize text
tokenizer = Tokenizer(num_words=max_words)
tokenizer.fit_on_texts(x_train)

# Convert text to sequences
x_train = tokenizer.texts_to_sequences(x_train)
x_test = tokenizer.texts_to_sequences(x_test)

# Pad sequences to uniform length
x_train = pad_sequences(x_train, maxlen=max_len)
x_test = pad_sequences(x_test, maxlen=max_len)

# 
print(f'Text example (tokenized): {x_train[0]}')
print(f'Label: {y_train[0]}')


In [ ]:
import tensorflow as tf
import keras_tuner as kt

# Create RNN model
def build_rnn_model(hp):
    model = tf.keras.models.Sequential()
    model.add(tf.keras.layers.Embedding(input_dim=max_words, output_dim=hp.Int('embedding_dim', 32, 128, step=32)))
    model.add(tf.keras.layers.LSTM(hp.Int('units', 32, 128, step=32)))
    model.add(tf.keras.layers.Dense(4, activation='softmax'))
    
    model.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])
    return model

# Use Grid Search via KerasTuner
tuner_rnn = kt.GridSearch(
    build_rnn_model,
    objective='val_accuracy',
    max_trials=5,  # Define number of trials (different tests)
    directory='../data/grid_search', 
    project_name='imdb_rnn'
)

# Run the search
tuner_rnn.search(x_train, y_train, epochs=5, validation_data=(x_test, y_test))

# Display best hyperparameters
best_hps_rnn = tuner_rnn.get_best_hyperparameters(num_trials=1)[0]

print(f'Best embedding dimension: {best_hps_rnn.get("embedding_dim")}')
print(f'Best number of LSTM units: {best_hps_rnn.get("units")}')

# Build and train the best model with optimized hyperparameters
best_rnn_model = tuner_rnn.hypermodel.build(best_hps_rnn)
best_rnn_model.fit(x_train, y_train, epochs=5, validation_data=(x_test, y_test))

In [ ]:
# RNN Model: Confusion Matrix
from sklearn.metrics import confusion_matrix, accuracy_score, precision_score, recall_score, f1_score
import seaborn as sns
import matplotlib.pyplot as plt

# #  best_rnn_model 
# best_rnn_model = tuner_rnn.hypermodel.build(best_hps_rnn)

# # 
# best_rnn_model.fit(x_train, y_train, epochs=5, validation_data=(x_test, y_test))

# Predict results from x_test
y_pred_rnn = best_rnn_model.predict(x_test).argmax(axis=1)

# Create Confusion Matrix
cm_rnn = confusion_matrix(y_test, y_pred_rnn)

# Display Confusion Matrix
plt.figure(figsize=(10, 8))
sns.heatmap(cm_rnn, annot=True, fmt='d', cmap='Blues')
plt.xlabel('Predicted')
plt.ylabel('True')
plt.show()

# Calculate Accuracy, Precision, Recall, and F1-score
accuracy_rnn = accuracy_score(y_test, y_pred_rnn)
precision_rnn = precision_score(y_test, y_pred_rnn, average='weighted')
recall_rnn = recall_score(y_test, y_pred_rnn, average='weighted')
f1_rnn = f1_score(y_test, y_pred_rnn, average='weighted')

# Display Accuracy, Precision, Recall, and F1-score
print(f'Accuracy: {accuracy_rnn:.4f}')
print(f'Precision: {precision_rnn:.4f}')
print(f'Recall: {recall_rnn:.4f}')
print(f'F1-score: {f1_rnn:.4f}')

In [ ]:
# RNN Model with Random Search
# Use KerasTuner for hyperparameter tuning
import tensorflow as tf
import keras_tuner as kt

# Create RNN model
def build_rnn_model(hp):
    model = tf.keras.models.Sequential()
    model.add(tf.keras.layers.Embedding(input_dim=max_words, output_dim=hp.Int('embedding_dim', 32, 128, step=32)))
    model.add(tf.keras.layers.LSTM(hp.Int('units', 32, 128, step=32)))
    model.add(tf.keras.layers.Dense(4, activation='softmax'))
    
    model.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])
    return model

# Use Random Search via KerasTuner
tuner_rnn = kt.RandomSearch(
    build_rnn_model,
    objective='val_accuracy',
    max_trials=5, 
    directory='../data/random_search', 
    project_name='imdb_rnn'
)

# Run the search
tuner_rnn.search(x_train, y_train, epochs=5, validation_data=(x_test, y_test))

# Display best hyperparameters
best_hps_rnn = tuner_rnn.get_best_hyperparameters(num_trials=1)[0]

print(f'Best embedding dimension: {best_hps_rnn.get("embedding_dim")}')
print(f'Best number of LSTM units: {best_hps_rnn.get("units")}')

# Build and train the best model with optimized hyperparameters
best_rnn_model = tuner_rnn.hypermodel.build(best_hps_rnn)
best_rnn_model.fit(x_train, y_train, epochs=5, validation_data=(x_test, y_test))

In [ ]:
# RNN Model: Confusion Matrix
from sklearn.metrics import confusion_matrix, accuracy_score, precision_score, recall_score, f1_score
import seaborn as sns
import matplotlib.pyplot as plt

# #  best_rnn_model 
# best_rnn_model = tuner_rnn.hypermodel.build(best_hps_rnn)

# # 
# best_rnn_model.fit(x_train, y_train, epochs=5, validation_data=(x_test, y_test))

# Predict results from x_test
y_pred_rnn = best_rnn_model.predict(x_test).argmax(axis=1)

# Create Confusion Matrix
cm_rnn = confusion_matrix(y_test, y_pred_rnn)

# Display Confusion Matrix
plt.figure(figsize=(10, 8))
sns.heatmap(cm_rnn, annot=True, fmt='d', cmap='Blues')
plt.xlabel('Predicted')
plt.ylabel('True')
plt.show()

# Calculate Accuracy, Precision, Recall, and F1-score
accuracy_rnn = accuracy_score(y_test, y_pred_rnn)
precision_rnn = precision_score(y_test, y_pred_rnn, average='weighted')
recall_rnn = recall_score(y_test, y_pred_rnn, average='weighted')
f1_rnn = f1_score(y_test, y_pred_rnn, average='weighted')

# Display Accuracy, Precision, Recall, and F1-score
print(f'Accuracy: {accuracy_rnn:.4f}')
print(f'Precision: {precision_rnn:.4f}')
print(f'Recall: {recall_rnn:.4f}')
print(f'F1-score: {f1_rnn:.4f}')